# MIDUS 3 — Final Reproducible Statistical Analysis
**Research:** *Statistical Analysis of Sex Differences in Cognitive Test Performance: Mean, Variance, and Distributional Comparisons*  
**Student:** Nurmukhammet Toigonbekov · MATDAIS-25 · 30 October 2026

This is a **single, self-contained Google Colab notebook** covering all seven project stages: data preparation, primary inferential models, quantiles and tails, sensitivity analyses, equivalence testing, distributional diagnostics, and the final audit/report. All values are **recomputed from the source data**; no hard-coded research results are used.

**Required inputs:** Only the official ICPSR Stata-format archives: `ICPSR_37095-V3.zip` (Cognitive Project, study 37095) and `ICPSR_36346-V7.zip` (Survey Project 1, study 36346). Files must be handled in accordance with ICPSR's terms of use.

**Run:** Upload the two ZIPs to the Colab Files pane, then select **Runtime → Run all**. Alternatively, the setup cell will ask for missing ZIP files. If uploading fails or an archive is corrupt, replace the original file and rerun. No external Python scripts are required.

**Privacy:** The notebook creates only aggregate tables, figures, and a research summary. No respondent-level data are exported.

**Interpretation:** The study compares scores in a **selected cohort**, not innate abilities, IQ, causal effects, or all U.S. adults. The recorded binary variable represents the study's original sex classification; it is not a general measure of gender identity.

## Setup — packages, random seed and source files

Colab already has NumPy, pandas, SciPy, statsmodels, and Matplotlib. The code validates **ZIP contents**, not merely filenames. Two valid source archives are required. To reproduce outside Colab, place them beside the notebook and execute all cells.

In [ ]:
from pathlib import Path
import io, json, zipfile, sys, platform
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm
import statsmodels
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 20261030
BOOT_PRIMARY = 2000   # target specified in the revised proposal
BOOT_QUANTILE = 1000
EQUIV_BOUND = 0.20    # reference-standardized BTACT units; NOT Hedges' g
ROOT = Path.cwd()
OUT = ROOT / 'MIDUS3_final_results'
OUT.mkdir(exist_ok=True)

required = {'ICPSR_37095-V3.zip':'37095-0001-Data.dta',
            'ICPSR_36346-V7.zip':'36346-0001-Data.dta'}
missing = [filename for filename in required if not (ROOT / filename).exists()]
if missing:
    try:
        from google.colab import files
        print('Upload missing archive(s):', missing)
        files.upload()
    except ImportError:
        raise FileNotFoundError(f'Missing {missing}; put both ZIP archives in {ROOT}')

for filename,member in required.items():
    file = ROOT/filename
    if not zipfile.is_zipfile(file):
        raise ValueError(f'{filename} is missing, incomplete, or not a ZIP archive. Re-upload it.')
    with zipfile.ZipFile(file) as z:
        matches = [p for p in z.namelist() if p.endswith(member)]
        if len(matches)!=1: raise ValueError(f'{filename}: expected exactly one {member}, found {matches}')
    print(f'✓ {filename} ({file.stat().st_size:,} bytes)')
print('Python:',sys.version.split()[0], 'pandas:',pd.__version__, 'scipy:', __import__('scipy').__version__,
      'statsmodels:',statsmodels.__version__)

## Stage 1 — Load, link, filter, and describe the sample

The principal analysis restricts to original MIDUS 3 cohort respondents (`C3STATUS == 1`) in components `SAMPLMAJ=1,2,3,4`. The 330 Milwaukee and 268 Refielding respondents are not silently pooled with the original cohort. `M2ID` is the unique respondent linkage key; `M2FAMNUM` indexes related respondents. `C1PRSEX=1` is coded male and `2` female. Factors with code `8` are missing, not exceptionally high cognitive scores.

**Age caveat:** `C1PRAGE` comes from the Project 1 survey, not the cognitive interview. We approximate age at cognitive testing as survey age plus the whole-month lag between interviews divided by 12. The actual date of birth is not in these public use files. This is an approximation and is tested in sensitivity analyses.

**Quality rule:** `C3TEF` includes a reaction-time task. The recommended **clean SGST** filter `C3TSFC == 1` is applied to the Executive Functioning main analysis; it is not imposed on Episodic Memory.

In [ ]:
def read_icpsr_stata(archive, suffix):
    with zipfile.ZipFile(ROOT/archive) as z:
        matches=[p for p in z.namelist() if p.endswith(suffix)]
        assert len(matches)==1, (archive,matches)
        return pd.read_stata(io.BytesIO(z.read(matches[0])),convert_categoricals=False)

cog=read_icpsr_stata('ICPSR_37095-V3.zip','37095-0001-Data.dta')
survey=read_icpsr_stata('ICPSR_36346-V7.zip','36346-0001-Data.dta')
assert cog.M2ID.is_unique and survey.M2ID.is_unique
original = cog.loc[cog.C3STATUS.eq(1) & cog.SAMPLMAJ.isin([1,2,3,4])].copy()
assert len(original)==2693, 'Unexpected number of original main-cohort observations'
cols=['M2ID','C1PRAGE','C1PBYEAR','C1PIDATE_YR','C1PIDATE_MO','C1PB1']
main=original.merge(survey[cols],on='M2ID',how='left',validate='one_to_one',
                    indicator=True,suffixes=('_cog','_survey'))
assert main._merge.eq('both').all() and main.M2FAMNUM.notna().all()
assert main.C1PRSEX.isin([1,2]).all()
main['male']=main.C1PRSEX.eq(1).astype(int)
main['sex_label']=np.where(main.male.eq(1),'Men','Women')
main['age_survey']=main.C1PRAGE_survey.astype(float)
main['lag_months']=(main.C3IDATE_YR-main.C1PIDATE_YR)*12+(main.C3IDATE_MO-main.C1PIDATE_MO)
assert main.lag_months.between(0,60).all()
main['age_test_approx']=main.age_survey+main.lag_months/12
main['age_birthyear_approx']=main.C3IDATE_YR-main.C1PBYEAR-0.5
main['education_code']=main.C1PB1.replace({97:np.nan,98:np.nan,99:np.nan}).astype(float)
main['education_level']=pd.cut(main.education_code,bins=[0,5,8,9,12],
    labels=['HS_or_below','Some_college_or_associate','Bachelors','Postgraduate'])
for y in ['C3TEM','C3TEF','C3TCOMP']:
    main[y]=main[y].replace(8,np.nan)
for task in ['C3TWLITU','C3TWLDTU']:
    main[task]=main[task].replace(98,np.nan)
    assert main[task].dropna().between(0,15).all()

LABELS={'C3TEM':'Episodic Memory','C3TEF':'Executive Functioning'}
def analytic_sample(y):
    d = main.loc[main.C3TSFC.eq(1)].copy() if y=='C3TEF' else main.copy()
    return d.dropna(subset=[y,'age_test_approx','male','M2FAMNUM']).copy()

age_table=main.groupby('sex_label').agg(n=('M2ID','size'),age_survey_mean=('age_survey','mean'),
             age_test_approx_mean=('age_test_approx','mean'),age_test_approx_median=('age_test_approx','median'),
             age_test_approx_sd=('age_test_approx','std'),interview_lag_months_mean=('lag_months','mean'))
age_smd=(main.loc[main.male.eq(1),'age_test_approx'].mean()-main.loc[main.male.eq(0),'age_test_approx'].mean())/np.sqrt(
    (main.loc[main.male.eq(1),'age_test_approx'].var(ddof=1)+main.loc[main.male.eq(0),'age_test_approx'].var(ddof=1))/2)
print('Total cognitive dataset:',len(cog),'| Survey records:',len(survey),'| Original matched:',len(main),
      '| Families:',main.M2FAMNUM.nunique())
print('Age SMD (male - female):',round(age_smd,5))
display(age_table.round(3))
print('Education missing:',main.education_level.isna().sum())
print('C3TEM analytic N:',len(analytic_sample('C3TEM')),'| C3TEF clean analytic N:',len(analytic_sample('C3TEF')))
assert len(analytic_sample('C3TEM'))==2687 and len(analytic_sample('C3TEF'))==2408
age_table.to_csv(OUT/'age_by_sex.csv')

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4.2))
for label,g in main.groupby('sex_label'):
    axs[0].hist(g.age_test_approx,bins=25,density=True,histtype='step',linewidth=1.8,label=label)
    x=np.sort(g.age_test_approx.to_numpy());y=np.arange(1,len(x)+1)/len(x)
    axs[1].step(x,y,where='post',label=label)
axs[0].set(title='Age at testing (approximate)',xlabel='Years',ylabel='Density')
axs[1].set(title='Age ECDF',xlabel='Years',ylabel='Cumulative proportion')
for ax in axs: ax.legend();ax.grid(alpha=.15)
fig.tight_layout();fig.savefig(OUT/'age_distributions.png',dpi=170)
plt.show()

## Stage 1 continued — Primary age-adjusted tests, Hedges' $g$, and family bootstrap

The **only confirmatory family** consists of two coefficients (male − female), one for each co-primary outcome, in a quadratic age-adjusted OLS model:

$$Y_i=\beta_0+\beta_1\mathrm{Male}_i+\beta_2(Age_i-\overline{Age})+\beta_3(Age_i-\overline{Age})^2+\varepsilon_i.$$

Standard errors are **cluster-robust by family**, and the two primary $p$-values are adjusted using **Holm**. Nominal 95% CIs are for each coefficient and are not joint multiplicity-adjusted confidence regions.

The descriptive **Hedges' $g$** uses the *unadjusted pooled within-sex SD*, whereas the primary regression coefficient is age-adjusted. Both can be reported, but they represent different estimands.

The age-adjusted **residual variance ratio** is $s^2_{M,\mathrm{resid}}/s^2_{F,\mathrm{resid}}$. Its interval uses whole-family resampling, model refitting in each replicate, confidence limits calculated on $\log(R)$, and exponentiation back to the ratio scale. These are nominal/exploratory 95% intervals.

In [ ]:
def model_fit(d,y,*,education=False,component=False,age='age_test_approx',interaction=False,cov='cluster'):
    required=[y,'male',age,'M2FAMNUM']
    if education: required+=['education_level']
    if component: required+=['SAMPLMAJ']
    dat=d.dropna(subset=required).copy()
    dat['age_c']=dat[age]-dat[age].mean()
    dat['age_c2']=dat.age_c**2
    formula=f'{y} ~ male + age_c + age_c2'
    if education: formula+=' + C(education_level)'
    if component: formula+=' + C(SAMPLMAJ)'
    if interaction: formula+=' + male:age_c'
    kwargs={'cov_type':'cluster','cov_kwds':{'groups':dat.M2FAMNUM}} if cov=='cluster' else {'cov_type':cov}
    fitted=smf.ols(formula,data=dat).fit(**kwargs)
    dat['residual']=fitted.resid
    return dat,fitted

def hedges_g(men,women):
    nm,nf=len(men),len(women)
    sp=np.sqrt(((nm-1)*np.var(men,ddof=1)+(nf-1)*np.var(women,ddof=1))/(nm+nf-2))
    return (1-3/(4*(nm+nf-2)-1))*(men.mean()-women.mean())/sp

def family_bootstrap_primary(d,y,reps=BOOT_PRIMARY,seed=SEED):
    codes,fams=pd.factorize(d.M2FAMNUM,sort=False)
    members=[np.flatnonzero(codes==k) for k in range(len(fams))]
    raw=d[[y,'male','age_test_approx']].to_numpy(float)
    rng=np.random.default_rng(seed)
    effects, gs, logr=[],[],[]
    for _ in range(reps):
        idx=np.concatenate([members[int(j)] for j in rng.integers(0,len(members),len(members))])
        a=raw[idx]; score=a[:,0]; male=a[:,1].astype(bool); age=a[:,2]
        if min(male.sum(),(~male).sum())<50: continue
        agec=age-age.mean();X=np.column_stack((np.ones(len(a)),male.astype(float),agec,agec**2))
        coeff=np.linalg.lstsq(X,score,rcond=None)[0];res=score-X@coeff
        vr=res[male].var(ddof=1)/res[~male].var(ddof=1)
        if vr<=0: continue
        effects.append(coeff[1]);gs.append(hedges_g(score[male],score[~male]));logr.append(np.log(vr))
    if len(effects)<.95*reps:raise RuntimeError('Insufficient valid family bootstrap draws')
    return {'valid':len(effects),'beta_ci':np.quantile(effects,[.025,.975]),
            'g_ci':np.quantile(gs,[.025,.975]),'vr_ci':np.exp(np.quantile(logr,[.025,.975]))}

primary_rows=[];primary_boot={};primary_models={}
for k,y in enumerate(LABELS):
    d=analytic_sample(y); d,fit=model_fit(d,y)
    men=d.loc[d.male.eq(1),y].to_numpy();women=d.loc[d.male.eq(0),y].to_numpy()
    res_m=d.loc[d.male.eq(1),'residual'];res_f=d.loc[d.male.eq(0),'residual']
    boot=family_bootstrap_primary(d,y,seed=SEED+101*k)
    primary_boot[y]=boot;primary_models[y]=(d,fit)
    p=fit.pvalues['male'];ci=fit.conf_int().loc['male']
    primary_rows.append({'outcome':y,'label':LABELS[y],'n':len(d),'men_n':len(men),'women_n':len(women),
      'families':d.M2FAMNUM.nunique(),'mean_men':men.mean(),'mean_women':women.mean(),
      'sd_men':men.std(ddof=1),'sd_women':women.std(ddof=1),
      'beta':fit.params['male'],'se_cluster':fit.bse['male'],'ci95_low':ci.iloc[0],'ci95_high':ci.iloc[1],
      'p_unadjusted':p,'hedges_g_raw':hedges_g(men,women),
      'hedges_g_ci_low':boot['g_ci'][0],'hedges_g_ci_high':boot['g_ci'][1],
      'vr_raw':men.var(ddof=1)/women.var(ddof=1),
      'vr_resid':res_m.var(ddof=1)/res_f.var(ddof=1),
      'vr_ci_low':boot['vr_ci'][0],'vr_ci_high':boot['vr_ci'][1],
      'bootstrap_valid':boot['valid']})
primary=pd.DataFrame(primary_rows)
primary['p_holm']=multipletests(primary.p_unadjusted,method='holm')[1]
primary['reject_H0_holm']=primary.p_holm.lt(.05)
primary.to_csv(OUT/'primary_results.csv',index=False)
display(primary[['label','n','men_n','women_n','beta','ci95_low','ci95_high','p_holm','hedges_g_raw','vr_resid','vr_ci_low','vr_ci_high']].round(5))

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4.2))
for i,r in primary.iterrows():
    y=1-i
    axs[0].errorbar(r.beta,y,xerr=[[r.beta-r.ci95_low],[r.ci95_high-r.beta]],fmt='o',capsize=5)
    axs[1].errorbar(r.vr_resid,y,xerr=[[r.vr_resid-r.vr_ci_low],[r.vr_ci_high-r.vr_resid]],fmt='o',capsize=5)
axs[0].axvline(0,ls='--',c='k',alpha=.6);axs[1].axvline(1,ls='--',c='k',alpha=.6)
for ax in axs:
    ax.set_yticks([1,0],['Episodic Memory','Executive Functioning']);ax.grid(axis='x',alpha=.2)
axs[0].set(title='Age-adjusted mean differences',xlabel='Men − women (95% cluster CI)')
axs[1].set(title='Residual variance ratios',xlabel='Male / female (95% family bootstrap CI)')
fig.tight_layout();fig.savefig(OUT/'primary_comparisons.png',dpi=170)
plt.show()

## Stage 2 — Exploratory quantiles and distribution tails

For each outcome, report differences at the pooled set of quantiles 10%, 25%, 50%, 75%, and 90%. Tail thresholds are the **pooled** sample's 10th and 90th percentiles and **are shared across sexes**. The analysis compares *proportions within sex* rather than raw numbers of participants. Ties at the boundary count using `<=` and `>=`; in tied data, the effective tail may exceed 10%.

Whole-family bootstrap provides **nominal** 95% intervals. These comparisons are unadjusted for covariates and exploratory, not causal or multiplicity-confirmed.

In [ ]:
Q=np.array([.1,.25,.5,.75,.9])
def tail_snapshot(scores,ismale):
    m=scores[ismale];f=scores[~ismale]
    if len(m)==0 or len(f)==0:raise ValueError('A group is missing')
    lo,hi=np.quantile(scores,[.1,.9])
    rates=np.array([(m<=lo).mean(),(f<=lo).mean(),(m>=hi).mean(),(f>=hi).mean()])
    ratios=np.array([rates[0]/rates[1] if rates[1]>0 else np.nan,
                     rates[2]/rates[3] if rates[3]>0 else np.nan])
    return np.quantile(m,Q)-np.quantile(f,Q),rates,ratios,(lo,hi)

def quantile_tail_boot(d,y,seed,reps=BOOT_QUANTILE):
    v=d[y].to_numpy(float); male=d.male.to_numpy(bool)
    initial=tail_snapshot(v,male)
    ids,fams=pd.factorize(d.M2FAMNUM,sort=False)
    members=[np.flatnonzero(ids==j) for j in range(len(fams))]
    rng=np.random.default_rng(seed);qdraws=[];ratio_draws=[]
    for _ in range(reps):
        idx=np.concatenate([members[int(j)] for j in rng.integers(0,len(members),len(members))])
        q,r,ratio,_=tail_snapshot(v[idx],male[idx])
        if np.all(np.isfinite(ratio)) and (ratio>0).all():
            qdraws.append(q);ratio_draws.append(np.log(ratio))
    if len(qdraws)<.95*reps:raise RuntimeError('Too few valid tail bootstrap draws')
    return initial,np.quantile(qdraws,[.025,.975],axis=0),np.exp(np.quantile(ratio_draws,[.025,.975],axis=0)),len(qdraws)

qrows=[];tailrows=[];qt_output={}
for k,y in enumerate(LABELS):
    d=analytic_sample(y);obs,q_ci,r_ci,nboot=quantile_tail_boot(d,y,SEED+k)
    diffs,rates,ratios,(lo,hi)=obs
    qt_output[y]=obs
    men=d.loc[d.male.eq(1),y].to_numpy();women=d.loc[d.male.eq(0),y].to_numpy()
    for i,q in enumerate(Q):
        qrows.append({'outcome':y,'percentile':int(q*100),'men':np.quantile(men,q),
                      'women':np.quantile(women,q),'difference_M_minus_F':diffs[i],
                      'ci95_lower':q_ci[0,i],'ci95_upper':q_ci[1,i]})
    for i,tail in enumerate(('Lower pooled 10%','Upper pooled 10%')):
        tailrows.append({'outcome':y,'tail':tail,'shared_cutoff':lo if i==0 else hi,
            'male_percent':rates[i*2]*100,'female_percent':rates[i*2+1]*100,
            'male_to_female_ratio':ratios[i], 'ci95_lower':r_ci[0,i],'ci95_upper':r_ci[1,i]})
quantiles=pd.DataFrame(qrows);tails=pd.DataFrame(tailrows)
quantiles.to_csv(OUT/'quantiles.csv',index=False);tails.to_csv(OUT/'tails.csv',index=False)
display(quantiles.round(3));display(tails.round(3))

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4.2))
for ax,y in zip(axs,LABELS):
    d=analytic_sample(y)
    for label,g in d.groupby('sex_label'):
        xx=np.sort(g[y].to_numpy()); yy=np.arange(1,len(xx)+1)/len(xx)
        ax.step(xx,yy,where='post',label=f'{label} (n={len(xx)})')
    ax.set(title=LABELS[y],xlabel='Reference-standardized BTACT score',ylabel='ECDF')
    ax.grid(alpha=.2);ax.legend()
fig.suptitle('Empirical cumulative distributions by sex')
fig.tight_layout();fig.savefig(OUT/'ecdf_both_outcomes.png',dpi=170)
plt.show()

## Stage 3 — Sensitivity analysis

We compare the prespecified primary estimate with alternatives: interview age instead of approximate testing age, education categories, sample-component indicators, Main RDD only, task-quality flags, and an exploratory sex-by-age interaction. A change in estimate across models is not evidence of a causal education effect. Restricting a sample can alter both the covariate distribution and the composition of the target group.

In [ ]:
sensitivity_rows=[]
for y in LABELS:
    d=analytic_sample(y)
    scenarios=[('Primary: approximate testing age',d,{}),
      ('Survey age',d,{'age':'age_survey'}),
      ('Approximate age from birth year',d,{'age':'age_birthyear_approx'}),
      ('Age and education',d,{'education':True}),
      ('Age and component',d,{'component':True}),
      ('Only Main RDD component',d.loc[d.SAMPLMAJ.eq(1)],{}),
      ('Sex × age interaction (contrast at mean age)',d,{'interaction':True}),
      ('HC3 instead of family-clustered SE',d,{'cov':'HC3'})]
    if y=='C3TEM':
        flag=(d.C3TWLIFP!=1)&(d.C3TWLDFP!=1)
        scenarios.append(('Exclude flagged word recall',d.loc[flag],{}))
    else:
        no_clean=main.dropna(subset=[y,'age_test_approx','M2FAMNUM'])
        scenarios.append(('No clean SGST filter — diagnostic only',no_clean,{}))
    for name,subset,kwargs in scenarios:
        dd,fit=model_fit(subset,y,**kwargs)
        ci=fit.conf_int().loc['male']
        sensitivity_rows.append({'outcome':y,'scenario':name,'n':len(dd),'sex_coef':fit.params['male'],
          'ci95_low':ci.iloc[0],'ci95_high':ci.iloc[1],'p_unadjusted':fit.pvalues['male'],
          'sex_age_interaction_p':fit.pvalues.get('male:age_c',np.nan)})
sensitivity=pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(OUT/'sensitivity.csv',index=False)
display(sensitivity.round(4))

## Stage 4 — Exploratory equivalence testing (TOST)

The *smallest effect size of interest* is fixed at $\pm 0.20$ **reference-standardized BTACT units**, not $g=0.20$. TOST separately tests whether the mean difference exceeds the lower boundary and falls below the upper one. We use the primary age-adjusted, family-clustered coefficient and its standard error.

TOST's two one-sided statistics are combined using the **maximum** of their one-sided $p$-values. The two cognitive domains make one **separate exploratory Holm-corrected equivalence family**, distinct from the confirmatory mean-difference tests. At unadjusted $\alpha=0.05$, 90% CI containment is equivalent to TOST; the final decision additionally requires Holm-adjusted $p<0.05$.

Failure to demonstrate equivalence is **not** proof of a large effect. A statistically significant, very small difference can also be statistically equivalent within a meaningful interval.

In [ ]:
tostrows=[]
for _,r in primary.iterrows():
    beta=r.beta;se=r.se_cluster;b=EQUIV_BOUND
    p_lower=norm.sf((beta+b)/se)
    p_upper=norm.cdf((beta-b)/se)
    tostrows.append({'outcome':r.outcome,'beta':beta,'ci90_low':beta-norm.ppf(.95)*se,
                     'ci90_high':beta+norm.ppf(.95)*se,'margin':b,
                     'tost_p_raw':max(p_lower,p_upper)})
tost=pd.DataFrame(tostrows)
tost['tost_p_holm']=multipletests(tost.tost_p_raw,method='holm')[1]
tost['equivalent_at_0p05']=tost.tost_p_holm.lt(.05)
tost.to_csv(OUT/'tost.csv',index=False)
display(tost.round(5))
# Alternative bounds are sensitivity descriptions, never an after-the-fact replacement for ±0.20.

In [ ]:
fig,ax=plt.subplots(figsize=(8,3.8))
ax.axvspan(-EQUIV_BOUND,EQUIV_BOUND,alpha=.12,label='Pre-specified equivalence bounds')
for i,r in tost.iterrows():
    yy=1-i
    ax.errorbar(r.beta,yy,xerr=[[r.beta-r.ci90_low],[r.ci90_high-r.beta]],fmt='o',capsize=5)
ax.axvline(0,ls='--',c='k',alpha=.6)
ax.set(yticks=[1,0],yticklabels=['Episodic Memory','Executive Functioning'],
       xlabel='Male − female age-adjusted difference (90% cluster CI)',title='Equivalence (TOST)')
ax.grid(axis='x',alpha=.2)
fig.tight_layout();fig.savefig(OUT/'tost_90ci.png',dpi=170)
plt.show()

## Stage 5 — Distributional & variability diagnostics

A median-centered Brown–Forsythe test can assess heterogeneity of variances, but its **usual independent-observation p-value is not valid** when relatives are present. We therefore also fit a **cluster-robust regression of absolute residual deviations** as a BF-style diagnostic. Its p-value is exploratory, not an additional primary hypothesis test.

The two-sample **Kolmogorov–Smirnov D** is recorded descriptively. Episodic-memory factor scores have many ties, so standard continuous KS p-value calibration is unsuitable. We also inspect variability in age tertiles and whether individual **raw** word-recall tasks reach their meaningful minimum (0) or maximum (15). The observed min/max of a standardized factor score must **not** be labelled a floor or ceiling by itself.

In [ ]:
diagrows=[];agerows=[];floorrows=[]
for y in LABELS:
    d,fit=primary_models[y]
    male=d.male.to_numpy(bool);m=d.loc[male,y].to_numpy();f=d.loc[~male,y].to_numpy()
    rm=d.loc[male,'residual'].to_numpy();rf=d.loc[~male,'residual'].to_numpy()
    bf=stats.levene(rm,rf,center='median')  # independent-person benchmark ONLY
    absdev=np.where(male,abs(d.residual-np.median(rm)),abs(d.residual-np.median(rf)))
    dd=d.assign(abs_dev=absdev)
    cluster_bf=smf.ols('abs_dev ~ male',data=dd).fit(cov_type='cluster',cov_kwds={'groups':dd.M2FAMNUM})
    diagrows.append({'outcome':y,'raw_KS_D':stats.ks_2samp(m,f).statistic,
      'age_adjusted_residual_KS_D':stats.ks_2samp(rm,rf).statistic,
      'n_unique_scores':d[y].nunique(),'BF_independent_p_descriptive_only':bf.pvalue,
      'BF_style_cluster_p_exploratory':cluster_bf.pvalues['male'],
      'BF_style_cluster_coef':cluster_bf.params['male']})
    agebands=pd.qcut(d.age_test_approx,3,duplicates='drop')
    for band,g in d.groupby(agebands,observed=True):
        gm=g.loc[g.male.eq(1),'residual'];gf=g.loc[g.male.eq(0),'residual']
        agerows.append({'outcome':y,'age_band':str(band),'n':len(g),
                        'variance_ratio_male_female':gm.var(ddof=1)/gf.var(ddof=1)})

for task in ['C3TWLITU','C3TWLDTU']:
    for sex,g in main.groupby('sex_label'):
        scores=g[task].dropna()
        floorrows.append({'task':task,'sex':sex,'n':len(scores),
                          'pct_floor_0':100*scores.eq(0).mean(),'pct_ceiling_15':100*scores.eq(15).mean()})
diagnostics=pd.DataFrame(diagrows)
by_age=pd.DataFrame(agerows)
floor_checks=pd.DataFrame(floorrows)
for name,tab in [('diagnostics',diagnostics),('variance_by_age',by_age),('word_recall_boundaries',floor_checks)]:
    tab.to_csv(OUT/f'{name}.csv',index=False)
display(diagnostics.round(5));display(by_age.round(3));display(floor_checks.round(3))

## Stage 6 — Audit: check reproducibility and confirmatory/exploratory separation

Assertions below catch mismatches in sample sizes, family linkage, score quality filters, Holm family and coefficient orientation. They check deterministic point estimates against the previously validated MIDUS calculations (rounded tolerances); bootstrap intervals may vary slightly because the final notebook runs **2,000**, rather than 1,200, primary resamples.

In [ ]:
EXPECTED={'C3TEM':{'n':2687,'beta':-0.47331301637491174,'vr':0.6674792901203814},
          'C3TEF':{'n':2408,'beta':0.1418518118926921,'vr':1.2792891760023155}}
audit=[]
def check(label,condition):
    audit.append((label,bool(condition)))
    if not condition:raise AssertionError(f'Validation failed: {label}')
check('2693 original cohort records',len(main)==2693)
check('2088 families',main.M2FAMNUM.nunique()==2088)
check('Unique participant IDs',main.M2ID.is_unique)
check('All primary participants linked to survey',main._merge.eq('both').all())
check('8 missing education entries',main.education_level.isna().sum()==8)
check('Exactly 2 Holm-adjusted primary tests',len(primary)==2 and primary.p_holm.notna().all())
check('Exactly 2 Holm-adjusted exploratory TOSTs',len(tost)==2 and tost.tost_p_holm.notna().all())
for _,r in primary.iterrows():
    check(f'{r.outcome}: analytic N',int(r.n)==EXPECTED[r.outcome]['n'])
    check(f'{r.outcome}: main contrast verified',np.isclose(r.beta,EXPECTED[r.outcome]['beta'],atol=1e-9))
    check(f'{r.outcome}: residual variance ratio verified',np.isclose(r.vr_resid,EXPECTED[r.outcome]['vr'],atol=1e-9))
    check(f'{r.outcome}: bootstrap valid',r.bootstrap_valid>=.95*BOOT_PRIMARY)
check('Executive Functioning uses clean SGST',analytic_sample('C3TEF').C3TSFC.eq(1).all())
print(f'Passed {sum(x[1] for x in audit)}/{len(audit)} checks')
pd.DataFrame(audit,columns=['check','passed']).to_csv(OUT/'audit_checks.csv',index=False)

## Stage 7 — Results summary and export

The tables below are generated **from computed dataframes**, not manually typed results. In Colab, open the left Files panel and download the `MIDUS3_final_results` folder files. Outputs include primary effects, quantiles, tail ratios, sensitivity comparisons, TOST, diagnostics, and figures. No individual records are exported.

Interpret cautiously: nominal confidence intervals for exploratory results are not multiple-comparison-adjusted. No inference supports an innate difference, broad IQ conclusions, or causal claims.

In [ ]:
print('FINAL PRIMARY TABLE')
display(primary[['label','n','men_n','women_n','beta','ci95_low','ci95_high','p_holm',
                 'hedges_g_raw','hedges_g_ci_low','hedges_g_ci_high','vr_resid','vr_ci_low','vr_ci_high']].round(4))
print('EQUIVALENCE')
display(tost[['outcome','beta','ci90_low','ci90_high','tost_p_holm','equivalent_at_0p05']].round(5))

# Build plain-text academic summary using the freshly computed data.
a,b=primary.iloc[0],primary.iloc[1];t=tost.set_index('outcome')
results_text = "\n".join([
    "RESULTS — MIDUS 3",
    f"The original cohort comprised {len(main):,} participants from {main.M2FAMNUM.nunique():,} families.",
    f"Episodic Memory (n={a.n:,}): adjusted male-female difference {a.beta:+.3f} (95% CI {a.ci95_low:+.3f} to {a.ci95_high:+.3f}; Holm-adjusted p={a.p_holm:.3g}); unadjusted Hedges g={a.hedges_g_raw:+.3f}.",
    f"Executive Functioning (n={b.n:,}, clean SGST): adjusted difference {b.beta:+.3f} (95% CI {b.ci95_low:+.3f} to {b.ci95_high:+.3f}; Holm-adjusted p={b.p_holm:.3g}); unadjusted Hedges g={b.hedges_g_raw:+.3f}.",
    f"Age-adjusted residual variance ratios M/F: {a.vr_resid:.3f} (95% family-bootstrap CI {a.vr_ci_low:.3f} to {a.vr_ci_high:.3f}) and {b.vr_resid:.3f} (95% CI {b.vr_ci_low:.3f} to {b.vr_ci_high:.3f}).",
    f"Exploratory TOST equivalence ±{EQUIV_BOUND:.2f} units: Episodic Memory {bool(t.loc['C3TEM','equivalent_at_0p05'])}; Executive Functioning {bool(t.loc['C3TEF','equivalent_at_0p05'])}.",
    "These are observational differences in domain-specific scores, not IQ differences or causal evidence."
])
(OUT/'results_summary.txt').write_text(results_text,encoding='utf8')
print(results_text)
print('Saved aggregate outputs to:',OUT.resolve())

### How to write the final paper (methodological reminders)

1. **Results:** explicitly separate the two *confirmatory* age-adjusted sex contrasts from all exploratory analyses. Report exact analytic sample sizes, family clusters, Holm-adjusted $p$, domain-specific effect sizes and uncertainty. State that Hedges' $g$ is not age-adjusted.
2. **Discussion:** compare the direction of average differences with Hughes et al. (2018), but do not claim a novel sex difference in means. Your contribution is domain-specific **between-person** residual variability and detailed distributional comparisons; unlike *within-person dispersion*, these quantities compare different respondents within each group.
3. **TOST:** the ±0.20 benchmark is a pragmatic reference-scale bound, **not a clinical cutoff**. It is possible to find a significant difference yet also establish equivalence relative to a nonzero bound.
4. **Caveats:** survey-to-cognition age is approximate; the original cohort is subject to attrition, selective survival, and recruitment bias; some participants are relatives; education may be confounding or part of a pathway; EF depends on the clean SGST inclusion criterion; episodic-memory scores are tied and recall tasks exhibit bounded-range artifacts. Quantiles and tails are *unadjusted* unless a separate age-adjusted analysis is explicitly performed.
5. **Reproducibility:** record dataset versions (ICPSR 37095 v3, 36346 v7), use official data access terms, retain random seed and software versions, and avoid exporting respondent-level data.